In [ ]:
# ============================================================
# 前置配置(2)：LangSmith 追踪（可选）
# ------------------------------------------------------------
# LangSmith 是 LangChain 官方的“可观测性/调试平台”。打开后，
# 每次调用链路（检索了哪些块、最终拼出的 Prompt、LLM 的输入输出、
# 耗时、token 用量）都会被记录，方便你排查“为什么答错了”。
# 它是可选的——不配也能正常跑 RAG。
#
# 注意：这几个 os.environ 必须在 import langchain 之前设置，
#       因为 langchain 导入时就会读取这些环境变量。
# ============================================================
import os

os.environ['LANGCHAIN_TRACING_V2'] = 'true'                          # 总开关：'true' 上报到 LangSmith；不想用改 'false' 即可
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com' # LangSmith 服务端地址（用官方云端就别改）
os.environ['LANGCHAIN_API_KEY'] = '<your-api-key>'                   # 你的 LangSmith Key（去 smith.langchain.com 申请后替换）


In [ ]:
# ============================================================
# 前置配置：OpenAI API Key（必需）
# ------------------------------------------------------------
# 后面两处会真实联网/计费地调用 OpenAI：
#   1) OpenAIEmbeddings —— 把文本转成向量（建索引、检索时用）
#   2) ChatOpenAI       —— 调用 GPT 生成最终答案
# 没有有效 Key，这两步会直接抛鉴权错误。
# ============================================================
os.environ['OPENAI_API_KEY'] = '<your-api-key>'   # 换成你的真实 OpenAI Key（sk-... 开头）


In [ ]:
# Part 1: Overview
# RAG quickstart

In [ ]:
# ============================================================
# Part 1 快速上手：一次性导入整条 RAG 流水线要用到的组件
# ------------------------------------------------------------
# RAG(检索增强生成) 的完整链条：
#   加载 → 切分 → 向量化建库 → 检索 → 拼 Prompt → LLM 生成
# 下面每个 import 对应其中一环。
#
# langchain 1.x 的两个关键变更（否则会 import 报红）：
#   1) `from langchain import hub` 已删除。原教程用 hub.pull("rlm/rag-prompt")
#      从云端拉现成的 RAG Prompt；这里改成在下方用本地 ChatPromptTemplate
#      自己写同样内容，既不用联网也不用 LangSmith Key。
#   2) RecursiveCharacterTextSplitter 从 langchain.text_splitter
#      迁到了独立包 langchain_text_splitters。
# ============================================================
import bs4                                                            # BeautifulSoup4：解析 HTML、抽取网页正文
from langchain_text_splitters import RecursiveCharacterTextSplitter   # 文本切分器：把长文档切成小块(chunk)
from langchain_community.document_loaders import WebBaseLoader         # 文档加载器：从 URL 抓网页内容
from langchain_community.vectorstores import Chroma                    # 向量数据库：存向量 + 相似度检索（新包：langchain_chroma）
from langchain_core.output_parsers import StrOutputParser             # 输出解析器：把 LLM 返回对象取成纯字符串
from langchain_core.prompts import ChatPromptTemplate                 # Prompt 模板：带 {占位符} 的提示词
from langchain_core.runnables import RunnablePassthrough              # LCEL 组件：把输入原样透传给下一步
from langchain_openai import ChatOpenAI, OpenAIEmbeddings             # OpenAI 封装：对话模型 + 文本嵌入模型


In [ ]:
#### INDEXING ####

# Load Documents

In [ ]:
# ================================================================
# 索引阶段 INDEXING —— 把外部文档变成“可被语义检索”的向量库
# 共 3 步：① 加载  ② 切分  ③ 向量化入库
# ================================================================

# ① 加载文档(Load)：用 WebBaseLoader 抓取这篇博客。
#    bs_kwargs + SoupStrainer 的作用：解析 HTML 时只保留
#    post-content / post-title / post-header 三类区域，
#    过滤掉导航栏、页脚等噪声，避免污染后续检索。
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()   # 返回 Document 列表，每个含 page_content(正文) 和 metadata(来源等)

# ② 切分(Split)：LLM 有上下文长度上限，且“小块”检索更精准，
#    所以把长文切成每块 1000 字符、相邻块重叠 200 字符。
#    重叠(overlap) 是为了防止一句话被切断后语义丢失——
#    让相邻两块共享一段内容，保住上下文连续性。
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# ③ 向量化 + 入库(Embed & Store)：
#    OpenAIEmbeddings 把每个文本块转成一个高维向量(语义指纹)，
#    Chroma 把这些向量存起来，并支持“给定查询向量，找最相近的块”。
#    这一步真实调用 OpenAI(按文本量计费)，需要有效的 OPENAI_API_KEY。
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()   # 把向量库包成“检索器”，之后 .invoke(问题) 就能取回相关块

# ================================================================
# 检索 + 生成 RETRIEVAL and GENERATION —— 用检索到的内容让 LLM 作答
# ================================================================

# ④ Prompt 模板：告诉 LLM“只根据给你的 context 回答，不知道就说不知道，
#    最多三句话”。{question}/{context} 是占位符，运行时被真实内容填入。
#    (原教程用 hub.pull("rlm/rag-prompt")，这里用本地模板复刻，内容一致。)
prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question}\n"
    "Context: {context}\n"
    "Answer:"
)

# ⑤ LLM：选 gpt-3.5-turbo。temperature=0 表示“几乎不随机”，
#    让回答稳定、可复现（RAG 通常希望忠于资料，不要自由发挥）。
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# ⑥ 后处理函数：检索器返回的是多个 Document 对象，
#    这里把它们的正文用空行拼成一整段纯文本，好塞进 Prompt 的 {context}。
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ⑦ 组装 RAG 链(LCEL)：LangChain 用 | 把各步串成“管道”，数据从左流到右。
#    第一步是一个字典，两个键会并行计算：
#      - context : 把用户问题交给 retriever 检索，再用 format_docs 拼成文本
#      - question: RunnablePassthrough() 把原始问题原样透传
#    然后 {context, question} 一起填进 prompt → 交给 llm → StrOutputParser 取纯文本。
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ⑧ 真正提问：把问题喂进整条链，返回一句基于博客内容的回答。
rag_chain.invoke("What is Task Decomposition?")


In [ ]:
#Part 2: Indexing
# Documents

In [ ]:
# ================================================================
# Part 2 索引原理：先用最小例子，直观理解“embedding + 相似度”
# ----------------------------------------------------------------
# 准备一句“问题”和一句“文档”。注意二者用词不同(pets vs cat)，
# 但语义相关——下面会看到它们的向量相似度并不低。
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."


In [ ]:
# ----------------------------------------------------------------
# 小工具：用 tiktoken 数 token
# ----------------------------------------------------------------
# LLM 不是按“字符/单词”而是按“token”计费和限长的。
# tiktoken 是 OpenAI 的分词器，能把文本切成 token 并数个数。
# 这在 RAG 里很实用：决定 chunk_size 多大、估算调用成本、
# 判断拼好的 Prompt 会不会超过模型上下文上限。
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """返回字符串在指定编码下的 token 数量。"""
    encoding = tiktoken.get_encoding(encoding_name)   # 取编码器；GPT-3.5/4 用 "cl100k_base"
    num_tokens = len(encoding.encode(string))         # encode() 把文本变成 token id 列表，取其长度
    return num_tokens


In [ ]:
# ----------------------------------------------------------------
# 把文本变成向量(embedding)
# ----------------------------------------------------------------
# embedding = 把一段文本映射成一串数字(向量)，让“语义”变成可计算的坐标。
# 语义相近的文本，向量在空间里也相近——这正是语义检索的基础。
from langchain_openai import OpenAIEmbeddings

embd = OpenAIEmbeddings()                     # 默认模型 text-embedding-ada-002（需 OPENAI_API_KEY）
query_result = embd.embed_query(question)     # 把“问题”这句话转成向量
document_result = embd.embed_query(document)  # 把“文档”这句话转成向量
len(document_result)                          # 向量维度：ada-002 是 1536 维


In [ ]:
# ----------------------------------------------------------------
# 手写余弦相似度：看清“检索”底层到底怎么比“像不像”
# ----------------------------------------------------------------
# 余弦相似度 = 两个向量夹角的余弦值，只看“方向”不看“长度”。
#   取值 -1 ~ 1：越接近 1 越相似，接近 0 表示无关。
#   公式：cos = (A·B) / (|A|·|B|) = 点积 / (两者模长之积)
# 向量库(如 Chroma)内部的“相似度检索”，本质就是这个计算(外加高效索引加速)。
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)     # 点积 A·B：对应位相乘再求和
    norm_vec1 = np.linalg.norm(vec1)     # |A|：向量1的模长(欧氏范数)
    norm_vec2 = np.linalg.norm(vec2)     # |B|：向量2的模长
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)   # pets↔cat 相似度会较高，验证“语义相近→向量相近”


In [ ]:
# ================================================================
# 索引 INDEXING（完整版，供 Part 3 检索使用）
# ----------------------------------------------------------------
# ① 加载：同 Part 1，抓同一篇博客并只保留正文三个区域。
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()
print(blog_docs)   # 打印看看抓到的正文


In [ ]:
# ② 切分：这次改用“按 token 计数”切分(from_tiktoken_encoder)，
#    每块 300 token、重叠 50 token。相比按字符切，按 token 切
#    更贴合 LLM 的真实计量方式。
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, chunk_overlap=50
)
splits = text_splitter.split_documents(blog_docs)   # 得到一批文档块


In [ ]:
# ③ 建索引：把切分块向量化后存入 Chroma，并包成检索器。
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()   # 默认检索器（不限定返回数量）


In [ ]:
#Part 3: Retrieval

In [ ]:
# ================================================================
# Part 3 检索 Retrieval
# ----------------------------------------------------------------
# 重新建一个检索器，但用 search_kwargs={"k": 1} 限定
# “只返回最相关的 1 个文档块”。k 就是“取回 Top-K 个最相似块”的 K。
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})


In [ ]:
# 执行检索：给一个问题，检索器返回最相关的文档块。
# langchain 1.x 已移除 retriever.get_relevant_documents，统一改用 .invoke()。
docs = retriever.invoke("What is Task Decomposition?")


In [ ]:
len(docs)   # 上面设了 k=1，这里应返回 1（只取回最相关的 1 块）


In [ ]:
#Part 4: Generation

In [ ]:
# ================================================================
# Part 4 生成 Generation
# ----------------------------------------------------------------
# 演示“Prompt 模板”这一环：把检索到的 context 和用户 question
# 拼进一段提示词，交给 LLM。
# langchain 1.x：ChatPromptTemplate 从 langchain.prompts 迁到了 langchain_core.prompts。
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 自定义 Prompt：明确要求“只根据下面的 context 回答”。
# {context}/{question} 是占位符，.invoke 时用真实值替换。
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
promp = ChatPromptTemplate.from_template(template)   # 生成 Prompt 对象（注意变量名是 promp）
print(promp)   # 打印看看模板结构(input_variables、messages 等)


In [ ]:
# LLM + 最简链：Prompt → LLM
# temperature=0：输出尽量确定、可复现（RAG 要忠于资料，不需要“创意”）。
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# 用 LCEL 的 | 把 Prompt 和 LLM 串成一条最小链（还没接输出解析器）。
chain = promp | llm


In [ ]:
# 运行这条最简链：手动把上一段检索到的 docs 当作 {context}，
# 问题当作 {question} 传入。返回的是一个消息对象(AIMessage)，
# 其中的 .content 才是文字答案。
chain.invoke({"context": docs, "question": "What is Task Decomposition?"})


In [ ]:
# ================================================================
# Part 4 收尾：组装“检索 + 生成”的完整 RAG 链
# ----------------------------------------------------------------
# 原教程这里用 from langchain import hub; hub.pull("rlm/rag-prompt") 拉官方 Prompt。
# langchain 1.x 已移除 hub，且 hub.pull 还需联网 + LangSmith Key，
# 所以这里用本地 ChatPromptTemplate 复刻同款“rlm/rag-prompt”，内容一致、免联网。
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt_hub_rag = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question}\n"
    "Context: {context}\n"
    "Answer:"
)

# 完整 RAG 链：
#   context  → retriever 检索到的文档(直接作为上下文)
#   question → RunnablePassthrough() 原样透传
#   → 填进 prompt_hub_rag → 交给 llm → StrOutputParser 取纯文本
# 注意：这里 context 直接用了 retriever 的原始输出(Document 列表)，
#       没经过 format_docs 拼接，是教程的简化写法(能跑，但不如 Part 1 规整)。
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_hub_rag
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")
